# Qwen Linear-Attention QKV Disentanglement Equivalence

Loads `Qwen/Qwen3.5-0.8B-Base`, replaces only `linear_attn.in_proj_qkv` linear layers with `DisentangledLinear`, disentangles them with `rank_fraction=0.2`, disables truncation by using full-rank forward reconstruction, and compares the original and disentangled models with the equivalence helper adapted from `simple_transformer_test.ipynb`.

In [1]:
import gc
import os
import sys
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from transformers import AutoModelForCausalLM, AutoTokenizer

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from disentangled_linear import DisentangledLinear

os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODEL_NAME = "Qwen/Qwen3.5-0.8B-Base"
DISENTANGLE_RANK_FRACTION = 0.1
DISENTANGLE_STEPS = 300
DISENTANGLE_LEARNING_RATE = 0.01
DISENTANGLE_SEED = 0

# Override this if needed: "cuda", "mps", or "cpu".
REQUESTED_DEVICE = None

if REQUESTED_DEVICE is not None:
    DEVICE = REQUESTED_DEVICE
elif torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

if DEVICE == "cuda":
    TORCH_DTYPE = torch.bfloat16
elif DEVICE == "mps":
    TORCH_DTYPE = torch.float16
else:
    TORCH_DTYPE = torch.float32

print(f"Using device={DEVICE}, dtype={TORCH_DTYPE}")

/Users/leonardevers/LLM_Sem/reward-density-quality/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device=mps, dtype=torch.float16


In [2]:
def clear_device_cache():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()


def load_qwen_model():
    load_kwargs = {
        "dtype": TORCH_DTYPE,
        "trust_remote_code": True,
    }
    if DEVICE == "cuda":
        load_kwargs["device_map"] = "cuda"

    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
    if DEVICE != "cuda":
        model = model.to(DEVICE)
    model.eval()
    model.config.use_cache = False
    return model


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading original {MODEL_NAME}")
original_model = load_qwen_model()
print("Loading disentangled copy")
disentangled_model = load_qwen_model()
clear_device_cache()

print(f"Loaded model class: {original_model.__class__.__name__}")
print(f"Parameters: {sum(p.numel() for p in original_model.parameters()):,}")

Loading original Qwen/Qwen3.5-0.8B-Base


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 320/320 [00:00<00:00, 5793.44it/s]


Loading disentangled copy


Loading weights: 100%|██████████| 320/320 [00:00<00:00, 5263.94it/s]


Loaded model class: Qwen3_5ForCausalLM
Parameters: 752,393,024


In [ ]:


def selected_linear_attn_qkv_targets(model, module_name, parameter_name):
    targets = []
    for module_name, module in model.named_modules():
        if not module_name.endswith(module_name):
            continue
        projection = getattr(module, parameter_name, None)
        if isinstance(projection, nn.Linear):
            targets.append((f"{module_name}.{parameter_name}", module, projection))
    return targets

In [ ]:

def disentangle_linear_attn_qkv_layers(targets, learning_rate, disentanglement_steps):
    rows = []
    start = time.perf_counter()

    for target_idx, (target_name, parent, projection) in enumerate(targets):
        if target_idx > 0:
            break
        print(f"[{target_idx + 1:02d}/{len(targets):02d}] Disentangling {target_name}")
        target_start = time.perf_counter()

        layer = DisentangledLinear.from_linear(
            projection,
            rank_fraction=DISENTANGLE_RANK_FRACTION,
        )
        layer.disentangle(
            steps=disentanglement_steps,
            learning_rate=learning_rate,
            seed=DISENTANGLE_SEED + target_idx,
            device="cpu",
            log_prefix=target_name,
            log_every=10,
        )

        layer.disentanglement_representation()

        optimization_keep_rank = layer.keep_rank
        layer.rank_fraction = 1.0  # Full-rank forward reconstruction: do not truncate.
        full_forward_keep_rank = layer.keep_rank
        setattr(parent, "in_proj_qkv", layer)

        elapsed = time.perf_counter() - target_start
        rows.append({
            "name": target_name,
            "shape_4d": layer.shape_4d,
            "optimization_keep_rank": optimization_keep_rank,
            "forward_keep_rank": full_forward_keep_rank,
            "max_rank": layer.max_rank,
            "seconds": elapsed,
        })
        print(
            f"    optimization keep_rank={optimization_keep_rank}, "
            f"forward keep_rank={full_forward_keep_rank}/{layer.max_rank}, "
            f"elapsed={elapsed:.1f}s"
        )
        clear_device_cache()

    total_elapsed = time.perf_counter() - start
    print(f"Finished {len(rows)} layers in {total_elapsed:.1f}s")
    print()
    return rows


module_names = ["self_attn", "linear_attn", "mlp"]
parameter_names = [["k_proj", "o_proj"], ["in_proj_z", "up_proj"], ["down_proj", "up_proj"]]
rank_fractions = [0.1, 0.2, 0.5]
disentanglement_steps = 300
disentangled_LR = [0.01, 0.02, 0.04]

module_name = module_names
parameter_name = parameter_names[0][0]  # "k_proj"

for lr in disentangled_LR:
    for module_num, parameter_name in enumerate(module_names):
        for parameter_name in parameter_names[module_num]:
            targets = selected_linear_attn_qkv_targets(disentangled_model, module_name=module_name, parameter_name=parameter_name)
            name, _, projection = targets[0]
            print(f"{name}: weight={tuple(projection.weight.shape)}, bias={projection.bias is not None}")
            print("rank fraction:", rank_fractions[0])
            print("disentanglement steps:", disentanglement_steps)
            print("disentanglment learning rate:", lr)
            disentanglement_rows = disentangle_linear_attn_qkv_layers(targets, lr, disentanglement_steps)


In [ ]:
def md_table(headers, rows):
    rows = [[str(cell) for cell in row] for row in rows]
    header = "| " + " | ".join(headers) + " |"
    sep = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return "\n".join([header, sep, *body])


summary_rows = [
    [
        row["name"],
        row["shape_4d"],
        row["optimization_keep_rank"],
        row["forward_keep_rank"],
        row["max_rank"],
        f"{row['seconds']:.1f}",
    ]
    for row in disentanglement_rows
]
display(Markdown(
    "## Disentangled layers\n"
    + md_table(
        ["layer", "shape_4d", "keep_rank during optimization", "keep_rank during forward", "max_rank", "seconds"],
        summary_rows,
    )
))

In [ ]:
# Adapted from notebooks/simple_transformer_test.ipynb.

def get_synthetic_batch(batch_size=2, seq_len=32, seed=0):
    torch.manual_seed(seed)
    vocab_size = getattr(original_model.config, "vocab_size", tokenizer.vocab_size)
    return {
        "input_ids": torch.randint(0, vocab_size, (batch_size, seq_len), device=DEVICE),
        "attention_mask": torch.ones(batch_size, seq_len, device=DEVICE, dtype=torch.long),
    }


def extract_outputs(model, batch):
    with torch.no_grad():
        outputs = model(
            **batch,
            output_hidden_states=True,
            output_attentions=False,
            return_dict=True,
            use_cache=False,
        )

    hidden = getattr(outputs, "last_hidden_state", None)
    if hidden is None and getattr(outputs, "hidden_states", None) is not None:
        hidden = outputs.hidden_states[-1]

    logits = getattr(outputs, "logits", None)
    attentions = getattr(outputs, "attentions", None)

    return hidden, logits, attentions


def assert_models_equivalent(
    model_a,
    model_b,
    atol_hidden=1e-5,
    atol_logits=1e-4,
    cosine_threshold=0.999,
    attention_cosine_threshold=0.999,
    check_attention=False,
):
    """
    Compares two models for functional equivalence.
    """

    model_a.eval()
    model_b.eval()

    batch = get_synthetic_batch()

    h_a, l_a, attn_a = extract_outputs(model_a, batch)
    h_b, l_b, attn_b = extract_outputs(model_b, batch)

    results = {}

    passed = True

    if h_a is not None and h_b is not None:
        hidden_mse = F.mse_loss(h_a.float(), h_b.float()).item()
        results["hidden_mse"] = hidden_mse

        if hidden_mse > atol_hidden:
            print(f"Hidden states differ too much: {hidden_mse:.6g}, with lower threshold: {atol_hidden}")
            passed = False

    if l_a is not None and l_b is not None:
        cos = F.cosine_similarity(
            l_a.float().flatten(),
            l_b.float().flatten(),
            dim=0,
        ).item()

        results["logit_cosine"] = cos

        if cos < cosine_threshold:
            print(f"Logits not similar enough: {cos:.6g}, with upper threshold: {cosine_threshold}")
            passed = False

        logit_mse = F.mse_loss(l_a.float(), l_b.float()).item()
        results["logit_mse"] = logit_mse

        if logit_mse > atol_logits:
            print(f"Logits differ too much: {logit_mse:.6g}, with lower threshold: {atol_logits}")

    if check_attention and attn_a is not None and attn_b is not None:
        attn_diffs = []

        for a, b in zip(attn_a, attn_b):
            attn_diffs.append((a.float() - b.float()).abs().mean().item())

        attn_diff = sum(attn_diffs) / len(attn_diffs)
        results["raw_attention_diff"] = attn_diff

        a = torch.cat([x.float().flatten() for x in attn_a])
        b = torch.cat([x.float().flatten() for x in attn_b])

        attn_cos = F.cosine_similarity(a, b, dim=0).item()
        results["attention_cosine"] = attn_cos

        if attn_cos < attention_cosine_threshold:
            print(f"Attention not similar enough: {attn_cos}, with lower threshold: {attention_cosine_threshold}")
            passed = False

    if passed:
        print("Models are equivalent within tolerance")
    else:
        print("Models are NOT equivalent within tolerance")

    for k, v in results.items():
        print(f"{k}: {v:.6g}")

    return results

In [ ]:
equivalence_results = assert_models_equivalent(
    original_model,
    disentangled_model,
    atol_hidden=1e-4 if TORCH_DTYPE != torch.float32 else 1e-5,
    atol_logits=1e-3 if TORCH_DTYPE != torch.float32 else 1e-4,
    cosine_threshold=0.999,
    check_attention=False,
)
equivalence_results